El objetivo de esta actividad es generar un dashboard de gráficas interactivas usando la librería Plotly para analizar los datos de su proyecto.



Instrucciones:

Generar un dashboard con gráficas interactivas o mapas utilizando la función 'make_subplots' de Plotly (mínimo 5 gráficas)
En el diseño de las gráficas deben considerar los elementos aprendidos en módulos anteriores (teoría del color, leyes de la Gestalt, percepción visual, semiología de las gráficas). Deben asegurarse que todas las gráficas sean explicables por si misma (títulos a ejes, a gráficas, a leyendas, etc).
Debe incluir botones, sliders y/o mapas en alguna de las gráficas.
Todas las gráficas deben ser autoexplicativas, es decir, incluir títulos, ejes y leyendas adecuados.
Exportar su dashboard en formato .html
Escribir un reporte explicando como utilizar su dashboard y que información es posible extraer de éste. Si lo creen necesario pueden incluir impresiones de su panel.

In [5]:
#librerias son un chingo
#descagar datos 
import yfinance as yf
#data frames
import pandas as pd 
#qur no me cancele y fincnce 
import time
#construccion de markowitx mvs
import numpy as np
from scipy.optimize import minimize
# a ver ahi les va la tarea 
#importamos las librerias plotly
import plotly.graph_objects as go
import plotly_express as px
import plotly 
from plotly.subplots import make_subplots
#mapa de los activos 
import geopandas as gdp
from geopy.geocoders import Nominatim
geolocator = Nominatim(user_agent="geo_stock_map")

In [6]:
ticker = "AAPL"

data = yf.Ticker(ticker)

hist = data.history(period="1y")

print(hist)

                                 Open        High         Low       Close  \
Date                                                                        
2024-05-03 00:00:00-04:00  185.772810  186.121171  181.801571  182.518188   
2024-05-06 00:00:00-04:00  181.493025  183.334321  179.572087  180.856033   
2024-05-07 00:00:00-04:00  182.587839  184.031021  180.467859  181.542770   
2024-05-08 00:00:00-04:00  181.990679  182.209646  180.597249  181.881195   
2024-05-09 00:00:00-04:00  181.702043  183.792180  181.254160  183.702606   
...                               ...         ...         ...         ...   
2025-04-28 00:00:00-04:00  210.000000  211.500000  207.460007  210.139999   
2025-04-29 00:00:00-04:00  208.690002  212.240005  208.369995  211.210007   
2025-04-30 00:00:00-04:00  209.300003  213.580002  206.669998  212.500000   
2025-05-01 00:00:00-04:00  209.080002  214.559998  208.899994  213.320007   
2025-05-02 00:00:00-04:00  206.089996  206.990005  202.160004  205.350006   

In [7]:
info = data.info
for key, value in info.items():
    print(f"{key}:{value}")

address1:One Apple Park Way
city:Cupertino
state:CA
zip:95014
country:United States
phone:(408) 996-1010
website:https://www.apple.com
industry:Consumer Electronics
industryKey:consumer-electronics
industryDisp:Consumer Electronics
sector:Technology
sectorKey:technology
sectorDisp:Technology
longBusinessSummary:Apple Inc. designs, manufactures, and markets smartphones, personal computers, tablets, wearables, and accessories worldwide. The company offers iPhone, a line of smartphones; Mac, a line of personal computers; iPad, a line of multi-purpose tablets; and wearables, home, and accessories comprising AirPods, Apple TV, Apple Watch, Beats products, and HomePod. It also provides AppleCare support and cloud services; and operates various platforms, including the App Store that allow customers to discover and download applications and digital content, such as books, music, video, games, and podcasts, as well as advertising services include third-party licensing arrangements and its own 

In [8]:


#mini portafolio para que yf no me mate o cancele o lo que sea
tickers = [ "AAPL","MSFT", "GOOGL", "AMZN", "NVDA"]

def construirdfs(tickers):
    data = {}
    info_list = []
    for ticker in tickers:
        #descargar los datos 
        tkr = yf.Ticker(ticker) #declarar el ticker
        df = tkr.history(period="1y") #obtener la información historica 
        #volverlo un solo df con multiindex 
        df.drop(columns=["Dividends", "Stock Splits"], errors='ignore', inplace=True)  #no relevantes en el de ahorita quiza madure en el futuro y lo incluya
        df.columns = pd.MultiIndex.from_product([[ticker], df.columns])  #crear el multiindex de mi df solito
        data[ticker] = df #guardarlo
        #el df de la información 
        info = tkr.info
        #los datos que me interesan
            #geocodificación para mapas haha 
        city = info.get('city', '')
        state = info.get('state', '')
        country = info.get('country', '')
        address = f"{city}, {state}, {country}"
        
        
        latitude, longitude = None, None
        try:
            location = geolocator.geocode(address)
            if location:
                latitude = location.latitude
                longitude = location.longitude
        except Exception as e:
            print(f"Error geocoding {ticker}: {e}")
        info_data = {
        'Ticker': ticker,
        'Name': info.get('longName'),
        'Sector': info.get('sector'),
        'Industry': info.get('industry'),
        'City': city,
        'State': state,
        'Country': country,
        'Exchange': info.get('exchange'),
        'Currency': info.get('currency'),
        'Address': address,
        'Latitude': latitude,
        'Longitude': longitude
        }

        info_list.append(info_data)
        time.sleep(0.5)  # para que yahoo no me cancele otra vez
    #hacer el merge solo una vez de los datos historicos 
    historicaldata = pd.concat(data.values(), axis=1)
    #volver mi larga lista de atributos un solo df 
    infodata = pd.DataFrame(info_list).set_index('Ticker')

    return historicaldata, infodata

In [9]:
historical_df, info_df = construirdfs(tickers)

print(historical_df.head())
print(info_df)

                                 AAPL                                      \
                                 Open        High         Low       Close   
Date                                                                        
2024-05-03 00:00:00-04:00  185.772810  186.121171  181.801571  182.518188   
2024-05-06 00:00:00-04:00  181.493025  183.334321  179.572087  180.856033   
2024-05-07 00:00:00-04:00  182.587839  184.031021  180.467859  181.542770   
2024-05-08 00:00:00-04:00  181.990679  182.209646  180.597249  181.881195   
2024-05-09 00:00:00-04:00  181.702043  183.792180  181.254160  183.702606   

                                            MSFT                          \
                              Volume        Open        High         Low   
Date                                                                       
2024-05-03 00:00:00-04:00  163224100  399.232008  404.065105  398.815177   
2024-05-06 00:00:00-04:00   78569700  405.662926  410.793737  403.291020   
202

In [10]:
historical_df.head()

AAPL                                      \
                                 Open        High         Low       Close   
Date                                                                        
2024-05-03 00:00:00-04:00  185.772810  186.121171  181.801571  182.518188   
2024-05-06 00:00:00-04:00  181.493025  183.334321  179.572087  180.856033   
2024-05-07 00:00:00-04:00  182.587839  184.031021  180.467859  181.542770   
2024-05-08 00:00:00-04:00  181.990679  182.209646  180.597249  181.881195   
2024-05-09 00:00:00-04:00  181.702043  183.792180  181.254160  183.702606   

                                            MSFT                          \
                              Volume        Open        High         Low   
Date                                                                       
2024-05-03 00:00:00-04:00  163224100  399.232008  404.065105  398.815177   
2024-05-06 00:00:00-04:00   78569700  405.662926  410.793737  403.291020   
2024-05-07 00:00:00-04:00   77305800  411.518162  411.528096  405.990359   
2024-05-08 00:00:00-04:00   45057100  405.077405  409.106641  403.628445   
2024-05-09 00:00:00-04:00   48983000  407.459243  409.592947  406.000380   

                                                 ...        AMZN              \
                                Close    Volume  ...        Open        High   
Date                                             ...                           
2024-05-03 00:00:00-04:00  403.578827  17446700  ...  186.990005  187.869995   
2024-05-06 00:00:00-04:00  410.406708  16996600  ...  186.279999  188.750000   
2024-05-07 00:00:00-04:00  406.238464  20018200  ...  188.919998  189.940002   
2024-05-08 00:00:00-04:00  407.429443  11792300  ...  187.440002  188.429993   
2024-05-09 00:00:00-04:00  409.195984  14689700  ...  188.880005  191.699997   

                                                                  NVDA  \
                                  Low       Close    Volume       Open   
Date                                                                     
2024-05-03 00:00:00-04:00  185.419998  186.210007  39172000  87.760163   
2024-05-06 00:00:00-04:00  184.800003  188.699997  34725300  89.360629   
2024-05-07 00:00:00-04:00  187.309998  188.759995  34048900  91.068074   
2024-05-08 00:00:00-04:00  186.389999  188.000000  26136400  89.453598   
2024-05-09 00:00:00-04:00  187.440002  189.500000  43368400  90.499262   

                                                                       
                                High        Low      Close     Volume  
Date                                                                   
2024-05-03 00:00:00-04:00  89.251669  87.011408  88.759834  398341000  
2024-05-06 00:00:00-04:00  92.189701  89.025740  92.109726  376203000  
2024-05-07 00:00:00-04:00  91.750848  88.981761  90.524254  437342000  
2024-05-08 00:00:00-04:00  91.164034  89.390615  90.382294  325721000  
2024-05-09 00:00:00-04:00  91.042083  88.202020  88.717850  378013000  

[5 rows x 25 columns]

In [11]:
info_df.head()

,Name,Sector,Industry,City,State,Country,Exchange,Currency,Address,Latitude,Longitude
Ticker,,,,,,,,,,,
AAPL,Apple Inc.,Technology,Consumer Electronics,Cupertino,CA,United States,NMS,USD,"Cupertino, CA, United States",37.322893,-122.032290
MSFT,Microsoft Corporation,Technology,Software - Infrastructure,Redmond,WA,United States,NMS,USD,"Redmond, WA, United States",47.669414,-122.123877
GOOGL,Alphabet Inc.,Communication Services,Internet Content & Information,Mountain View,CA,United States,NMS,USD,"Mountain View, CA, United States",37.389389,-122.083210
AMZN,"Amazon.com, Inc.",Consumer Cyclical,Internet Retail,Seattle,WA,United States,NMS,USD,"Seattle, WA, United States",47.603832,-122.330062
NVDA,NVIDIA Corporation,Technology,Semiconductors,Santa Clara,CA,United States,NMS,USD,"Santa Clara, CA, United States",37.354113,-121.955174


In [12]:
#nunca volver a correr lo de arriba a menos que sea absolutamente necesario o me mato 
#una construcción sencilla markowitz


#estas se ponen a fuera porque son utiles en mas de una cosa van en PORTFOLIO el papá



def portfolio_variance(weights, cov_matrix):
    return weights.T @ cov_matrix @ weights

def portfolio_return(weights, mean_returns):
    return weights.T @ mean_returns


def MeanVariance(historical_df):
    #get the log close prices to construct the porfolio 
    closeprices = historical_df.xs('Close',axis = 1 , level = 1)
    #get the log returns 
    logreturns = np.log(closeprices / closeprices.shift(1)).dropna()

    #neet to know for optimizer 
    returnsxbar = logreturns.mean()
    covmatrix = logreturns.cov()
    tickers =  historical_df.columns.get_level_values(0).unique()
    n = len(tickers)

    #constraints 
    target_return = 0.2 / 252  # 
    constraints = [
    {'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1},  # sum of weights = 1
    {'type': 'eq', 'fun': lambda weights: portfolio_return(weights,returnsxbar) - target_return}
    ]
    bounds = tuple((0, 1) for _ in range(n)) #allowed values in the solution 
    initial_weights = np.ones(n) / n  # equally weighted

    result = minimize(
            portfolio_variance,
            initial_weights,
            args=(covmatrix,),
            method='SLSQP',
            bounds=bounds,
            constraints=constraints
        )

    if result.success:
        mvweights = result.x
        portvar = portfolio_variance(mvweights, covmatrix)
        portvolatility = np.sqrt(portvar)
        port_ret = portfolio_return(mvweights, returnsxbar)
        sharpe_ratio = port_ret / portvolatility
        return mvweights
    else:
        print(" fallo la optimización:", result.message)
        return None






In [13]:
#sharpe ratio minimization is the next one i need to knock 

# versión mejorada de la pasada honestamente la pasda la podria borrar 
def portfolio_variance(weights, cov_matrix):
    return weights.T @ cov_matrix @ weights

def portfolio_return(weights, mean_returns):
    return weights.T @ mean_returns

def sharpe_ratio(weights, mean_returns, cov_matrix):
    port_ret = portfolio_return(weights, mean_returns)
    
    port_var = portfolio_variance(weights, cov_matrix)
    port_volatility = np.sqrt(port_var)
    
    
    return -port_ret / port_volatility 

def MeanVarianceSharpe(historical_df):
    closeprices = historical_df.xs('Close', axis=1, level=1)
   
    logreturns = np.log(closeprices / closeprices.shift(1)).dropna()

    mean_returns = logreturns.mean()
    cov_matrix = logreturns.cov()
    tickers = historical_df.columns.get_level_values(0).unique()
    n = len(tickers)

    constraints = [{'type': 'eq', 'fun': lambda weights: np.sum(weights) - 1}]
    

    bounds = tuple((0, 1) for _ in range(n)) 
    
    # Initial equal weights
    initial_weights = np.ones(n) / n

    # Minimize negative Sharpe ratio (maximize Sharpe ratio)
    result = minimize(
        sharpe_ratio,
        initial_weights,
        args=(mean_returns, cov_matrix),
        method='SLSQP',
        bounds=bounds,
        constraints=constraints
    )

    if result.success:
        optimal_weights = result.x
        port_var = portfolio_variance(optimal_weights, cov_matrix)
        port_volatility = np.sqrt(port_var)
        port_ret = portfolio_return(optimal_weights, mean_returns)
        sharpe = -result.fun  # The result is negative because we minimized the negative Sharpe ratio

        return {
            'weights': optimal_weights,
            'expected_return': port_ret,
            'volatility': port_volatility,
            'sharpe_ratio': sharpe
        }
    else:
        print("Optimization failed:", result.message)
        return None


In [14]:
mvs = MeanVarianceSharpe(historical_df)
print("\n📊 Resultados del Portafolio Óptimo (Sharpe Máximo):")
print("-" * 50)
for ticker, weight in zip(historical_df.columns.get_level_values(0).unique(), mvs['weights']):
    print(f"{ticker:<10}: {weight:.2%}")
print("-" * 50)
print(f"📈 Retorno esperado: {mvs['expected_return']:.4f}")
print(f"📉 Volatilidad     : {mvs['volatility']:.4f}")
print(f"⚖️  Sharpe Ratio   : {mvs['sharpe_ratio']:.4f}")


📊 Resultados del Portafolio Óptimo (Sharpe Máximo):
--------------------------------------------------
AAPL      : 57.22%
MSFT      : 0.00%
GOOGL     : 0.00%
AMZN      : 0.00%
NVDA      : 42.78%
--------------------------------------------------
📈 Retorno esperado: 0.0007
📉 Volatilidad     : 0.0233
⚖️  Sharpe Ratio   : 0.0304


In [15]:
#como ver los plots de plotly
import plotly.io as pio #intput and output modules how are plots rendered 
pio.renderers.default = 'browser'#here i tell them its gonna happen on the brwoser 

#  MAPA

In [16]:
#soy el mapa soy el mapa
#este solo muerta lso activos en el mapa
def donde(info_df):
    fig = px.scatter_geo(info_df,
                        lat='Latitude',
                        lon='Longitude',
                        hover_name='Name',
                        text=info_df.index,
                        title="Ubicación de Sedes Corporativas",
                        projection='natural earth')
    return fig
    #fig.show()


In [17]:
def densidadporpais(mvs, info_df, historical_df):
    tickers = historical_df.columns.get_level_values(0).unique()
    weights = mvs['weights']

    # df para pesos de los ticker 
    weights_df = pd.DataFrame({
        'Ticker': tickers, # columna 1 
        'Weight': weights #columna 2 
    })

    #agreafar al df el pais 
    weights_df = weights_df.merge(info_df[['Country']], left_on='Ticker', right_index=True)

    # Crear columna de tooltip con tickers y pesos por país
    def format_hover_text(df):
        return "<br>".join([
            f"{row['Ticker']}: {row['Weight']:.2%}"
            for _, row in df.iterrows()
        ])

    # Agrupar por país y construir los textos de hover
    grouped = weights_df.groupby('Country')
    country_weights = grouped['Weight'].sum().reset_index()
    country_weights['Assets'] = grouped.apply(format_hover_text).values

    # Crear el mapa coroplético con hover_data personalizado
    fig = px.choropleth(
        country_weights,
        locations='Country',
        locationmode='country names',
        color='Weight',
        color_continuous_scale='Blues',
        range_color=(0, 1), 
        title='Portfolio distribution per country',
        labels={'Weight': 'Peso del Portafolio'},
        hover_name='Country',
        hover_data={'Assets': True, 'Weight': ':.2%'},
        height=600
    )

    fig.update_layout(
        geo=dict(showframe=False, showcoastlines=True),
        coloraxis_colorbar=dict(title="Peso Total (%)")
    )

    return fig
    #fig.show()



In [18]:
def densidadeua(mvs, info_df, historical_df):
    tickers = historical_df.columns.get_level_values(0).unique()
    weights = mvs['weights']

    # df para pesos de los ticker 
    weights_df = pd.DataFrame({
        'Ticker': tickers, # columna 1 
        'Weight': weights #columna 2 
    })

    #agreafar al df el en este caso el estado  
    weights_df = weights_df.merge(info_df[['State']], left_on='Ticker', right_index=True)

    # Crear columna de tooltip con tickers y pesos por país
    def format_hover_text(df):
        return "<br>".join([
            f"{row['Ticker']}: {row['Weight']:.2%}"
            for _, row in df.iterrows()
        ])

    # Agrupar por país y construir los textos de hover
    grouped = weights_df.groupby('State')
    country_weights = grouped['Weight'].sum().reset_index()
    country_weights['Assets'] = grouped.apply(format_hover_text).values

    # Crear el mapa coroplético con hover_data personalizado
    fig = px.choropleth(
        country_weights,
        locations='State',
        locationmode='USA-states',
        color='Weight',
        color_continuous_scale='Blues',
        range_color=(0, 1), 
        title='Portfolio distribution per US country',
        labels={'Weight': 'Peso del Portafolio'},
        hover_name='State',
        hover_data={'Assets': True, 'Weight': ':.2%'},
        height=600
    )

    fig.update_layout(
        geo=dict(
            scope ="usa",
            showframe=False, showcoastlines=True),
        coloraxis_colorbar=dict(title="Peso Total (%)")
    )
    return fig
    #fig.show()



# DATOS HISTORICOS 

In [19]:
#grafica de los datos histericos de los activos en un menu desplegable y un slider 

def historictimeseries(historical_df):
    tickers = historical_df.columns.get_level_values(0).unique() #las acciones 
    variables = historical_df.columns.get_level_values(1).unique() #las variables
    index = historical_df.index # las fechas 


    fig = go.Figure()
    #crea lla linea 
    traces = []
    for ticker in tickers:
        for var in variables:
            fig.add_trace(go.Scatter(
                x=index,
                y=historical_df[(ticker, var)],
                mode='lines',
                name=f'{ticker} - {var}',
                visible=False  
            ))
            traces.append((ticker, var))


    fig.data[0].visible = True


    ticker_buttons = [
        dict(
            label=ticker,
            method='update',
            args=[
                {'visible': [t == ticker and v == variables[0] for (t, v) in traces]},
                
            ]
        ) for ticker in tickers
    ]



    variable_buttons = [
        dict(
            label=var,
            method='update',
            args=[
                {'visible': [t == tickers[0] and v == var for (t, v) in traces]},
            ]
        ) for var in variables
    ]


    # Layout con dos menús desplegables
    fig.update_layout(
        annotations=[
        dict(
            text="Note: 'Volume' represents the number of titles traded, not a price.",
            xref="paper", yref="paper",
            x=0.6, y=1.20,
            showarrow=False,
            font=dict(size=11, color="gray"),
            align="left"
        )
    ],
        updatemenus=[
            dict(
                active=0,
                buttons=ticker_buttons,
                x=0.01, xanchor='left',
                y=1.15, yanchor='top',
                direction='down',
                showactive=True,
                type='dropdown',
                name='Select Ticker'
            ),
            dict(
                active=0,
                buttons=variable_buttons,
                x=0.2, xanchor='left',
                y=1.15, yanchor='top',
                direction='down',
                showactive=True,
                type='dropdown',
                name='Select Variable'
            )
        ],

        title=dict(
            text=f'Historical Data Time Series',
            xanchor='center', x=0.5,
            y=0.96, yanchor='middle'
        ),
        xaxis=dict(
            rangeslider=dict(visible=True, bgcolor='#f5f5f5' ,
                            bordercolor='red', borderwidth=2, thickness=0.2),
            type='date',
            title='Date'
        ),
        yaxis=dict(title='Price'),
        

        
        height=600
    )

    return fig
    #fig.show()




# SCATTER PLOT

In [20]:
def bubbleplot(historical_df, mvs):

    tickers = historical_df.columns.get_level_values(0).unique()
    weights = mvs['weights']
    
    closeprices = historical_df.xs('Close', axis=1, level=1)
    returns = closeprices / closeprices.shift(1).dropna()

    ereturns = returns.mean()
    vols = returns.std()


    bubbledf = pd.DataFrame({
        'Ticker': tickers, 
        'Returns': ereturns.values,
        'Volatility':vols.values,
        'Weight': weights 

    })
    
    fig = go.Figure(data=go.Scatter(
    x=bubbledf['Returns'],
    y=bubbledf['Volatility'],
    mode='markers',
    marker=dict(
        size=bubbledf['Weight']* 100,
        sizemode='area',
        sizeref=2.*max(bubbledf['Weight']* 100)/(100**2),  
        sizemin=4,
        color='skyblue',
        line=dict(width=1, color='DarkSlateGrey')
    ),
     text=bubbledf['Ticker'] 
     
      ) )
    
    fig.update_layout(
        title = 'Bubble Plot Returns vs Volatility',
        xaxis_title='Expected Returns',
        yaxis_title='Volatility',
        hovermode='closest'

    )
    return fig
    #fig.show()



In [21]:
fig1 = donde(info_df)
fig2 = densidadporpais(mvs, info_df, historical_df)
fig3 = densidadeua(mvs, info_df, historical_df)
fig4 = historictimeseries(historical_df)
fig5 = bubbleplot(historical_df,mvs)


C:\Users\herie\AppData\Local\Temp\ipykernel_5080\1378001908.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

C:\Users\herie\AppData\Local\Temp\ipykernel_5080\44128892.py:24: DeprecationWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.



In [24]:
fig1.show()
fig2.show()
fig3.show()
fig4.show()
fig5.show()



# Guardar los gráficos como archivos HTML
fig1.write_html("fig1.html")
fig2.write_html("fig2.html")
fig3.write_html("fig3.html")
fig4.write_html("fig4.html")
fig5.write_html("fig5.html")

# Hacer el Mosaico

In [23]:
fig = make_subplots(rows=3, cols=2,
                    specs=[[{'type': 'geo'}, {'type': 'geo'}],
                           [{'type': 'geo'}, {'type': 'xy'}],
                           [{'type': 'xy', 'colspan':2}, None]],
                    subplot_titles=(fig2.layout.title.text, fig3.layout.title.text,
                                   fig1.layout.title.text, fig5.layout.title.text,
                                   fig4.layout.title.text),
                    vertical_spacing=0.1,  # Aumentamos el espaciado vertical para los títulos
                    horizontal_spacing=0.05)

# Add the traces from the original figures to the main figure
for trace in fig2.data:
    fig.add_trace(trace, row=1, col=1)
for trace in fig3.data:
    fig.add_trace(trace, row=1, col=2)
for trace in fig1.data:
    fig.add_trace(trace, row=2, col=1)
for trace in fig5.data:
    fig.add_trace(trace, row=2, col=2)

# Set the geo scopes for the maps
fig.layout.geo.scope = "world"
fig.layout.geo2.scope = "usa"
fig.layout.geo3.scope = "world"

# Set coloraxis for choropleth maps
fig.data[0].coloraxis = 'coloraxis1'
fig.data[1].coloraxis = 'coloraxis2'

tickers = historical_df.columns.get_level_values(0).unique()
variables = historical_df.columns.get_level_values(1).unique()

historical_trace_indices = []  # Keep track of the trace indices for the historical data
start_index = len(fig.data)  # Index where historical traces will start

# Add historical time series traces with proper naming
for ticker in tickers:
    for var in variables:
        trace_data = historical_df[(ticker, var)]
        trace = go.Scatter(
            x=historical_df.index,
            y=trace_data,
            mode='lines',
            name=f'{ticker} - {var}',
            visible=False
        )
        fig.add_trace(trace, row=3, col=1)
        historical_trace_indices.append(len(fig.data) - 1)

if historical_trace_indices:
    fig.data[historical_trace_indices[0]].visible = True

ticker_buttons = []
for ticker in tickers:
    visibility = [True] * start_index
    for i, idx in enumerate(historical_trace_indices):
        ticker_name = fig.data[idx].name.split(' - ')[0]
        visibility.append(ticker_name == ticker)

    ticker_buttons.append(dict(
        label=ticker,
        method='update',
        args=[{'visible': visibility}]
    ))

variable_buttons = []
for var in variables:
    visibility = [True] * start_index
    for i, idx in enumerate(historical_trace_indices):
        var_name = fig.data[idx].name.split(' - ')[1]
        visibility.append(var_name == var)

    variable_buttons.append(dict(
        label=var,
        method='update',
        args=[{'visible': visibility}]
    ))

fig.update_layout(
    coloraxis1=dict(
        colorbar_title="Weight",
        colorscale='Blues',
        cmin=0,
        cmax=1,  # Cambiado aquí para asegurar el rango de 0-1
        colorbar=dict(
            x=0.45,
            y=0.85,
            len=0.3
        )
    ),
    coloraxis2=dict(
        colorbar_title="Weight",
        colorscale='Blues',
        cmin=0,
        cmax=1,  # Cambiado aquí para asegurar el rango de 0-1
        colorbar=dict(
            x=0.95,
            y=0.85,
            len=0.3
        )
    ),
    updatemenus=[
        dict(
            active=0,
            buttons=ticker_buttons,
            x=0.01,
            xanchor='left',
            y=0.25,
            yanchor='top',
            direction='down',
            showactive=False,
            type='dropdown',
            name='Select Ticker'
        ),
        dict(
            active=0,
            buttons=variable_buttons,
            x=0.2,
            xanchor='left',
            y=0.25,
            yanchor='top',
            direction='down',
            showactive=False,
            type='dropdown',
            name='Select Variable'
        )
    ],
    annotations=[
        dict(
            text="Note: 'Volume' represents the number of titles traded, not a price.",
            xref="paper",
            yref="paper",
            x=0.6,
            y=0.25,
            showarrow=True,
            font=dict(size=11, color="gray"),
            align="left"
        )
    ]
)

fig.update_xaxes(
    rangeslider=dict(
        visible=True,
        bgcolor='#f5f5f5',
        bordercolor='red',
        borderwidth=2,
        thickness=0.1
    ),
    type='date',
    title='Date',
    row=3,
    col=1
)

fig.update_yaxes(title='Price', row=3, col=1)

fig.update_layout(
    height=1200,
    width=1000,
    title_text='Portfolio analysis',  # Título principal
    title_font=dict(size=20),
    font=dict(size=12),  # Tamaño de fuente general
    showlegend=True
)

fig.update_layout(
    annotations=[
        dict(
            text= fig2.layout.title.text,
            xref="paper",
            yref="paper",
            x=0.225,  # Posición x (centrada para columna 1)
            y=1.0,  # Posición y (arriba de la gráfica)
            showarrow=False,
            font=dict(size=14)
        ),
        # Título para la gráfica en fila 1, columna 2
        dict(
            text=fig1.layout.title.text,
            xref="paper",
            yref="paper",
            x=0.775,  # Posición x (centrada para columna 2)
            y=1.0,  # Posición y (arriba de la gráfica)
            showarrow=False,
            font=dict(size=14)
        ),
        # Continúa con las demás gráficas...
        # Para fila 2, columna 1
        dict(
            text=fig3.layout.title.text,
            xref="paper",
            yref="paper",
            x=0.225,
            y=0.66,  # Ajusta según la distribución de tu dashboard
            showarrow=False,
            font=dict(size=14)
        ),
        dict(
            text=fig5.layout.title.text,
            xref="paper",
            yref="paper",
            x=0.775,  # Posición x (centrada para columna 2)
            y=0.66,  # Posición y (arriba de la gráfica)
            showarrow=False,
            font=dict(size=14)
        ),
        dict(
            text=fig4.layout.title.text,
            xref="paper",
            yref="paper",
            x=0.5,  # Posición x (centrada para columna 2)
            y=0.3,  # Posición y (arriba de la gráfica)
            showarrow=False,
            font=dict(size=14)
        ),
    ]
)

fig.show()

# guardar el archivo html
fig.write_html("dashboard.html", include_plotlyjs='cdn')  # Cambia 'dashboard.html' al nombre que desees